In [1]:
import pandas as pd
import numpy as np 
import duckdb 

In [3]:
con = duckdb.connect('../data/database/ne_pipeline.db')


In [4]:
con.execute("""
            SHOW TABLES
""").fetchall()


[('train_delay',), ('weather',)]

In [ ]:
train = con.sql("""
        SELECT * FROM train_delay
""")
weather = con.sql("""
        SELECT * FROM weather
""")

merged = con.sql("""
        SELECT * EXCLUDE(w.train_no,w.date) 
        FROM  train_delay td
        inner join weather w
        on td.date = w.date
 """)

In [ ]:
merged.to_df().info()

<class 'pandas.DataFrame'>
RangeIndex: 1085053 entries, 0 to 1085052
Data columns (total 13 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   train_no                   1085053 non-null  int32         
 1   date                       1085053 non-null  datetime64[us]
 2   station                    1085053 non-null  str           
 3   delay_minutes              974110 non-null   float32       
 4   temperature_2m_max         1085053 non-null  float64       
 5   temperature_2m_min         1085053 non-null  float64       
 6   temperature_2m_mean        1085053 non-null  float64       
 7   precipitation_sum          1085053 non-null  float64       
 8   rain_sum                   1085053 non-null  float64       
 9   wind_speed_10m_max         1085053 non-null  float64       
 10  wind_gusts_10m_max         1085053 non-null  float64       
 11  relative_humidity_2m_mean  1085053 non-null  int

In [16]:
train['delay_minutes'].value_counts()

delay_minutes
0.0      8162
1.0      4425
2.0      2501
3.0      1946
8.0      1892
         ... 
802.0       1
877.0       1
937.0       1
642.0       1
727.0       1
Name: count, Length: 836, dtype: int64

In [ ]:
'''[ASSERT] Every date in train data exists in weather data, and vice versa'''

def assert_dates(df_ts,df_w):
    ts_dates = set(df_ts['date'].dt.date)
    w_dates = set(df_w['date'].dt.date)
    #set difference
    missing_in_weather = ts_dates - w_dates
    missing_in_train = w_dates - ts_dates

    assert not missing_in_weather and not missing_in_train,(
        f"Mismatch!\n"
        f"Missing in weather: {missing_in_weather}\n"
        f"Extra in weather: {missing_in_train}"
    )

''' [ASSERT] a lossless merge of two dataset '''
def assert_lossless_join(df_merged,df_ts):
    #row mismatch
    assert df_merged.shape[0] ==  df_ts.shape[0]
    #duplicate rows
    assert df_merged.duplicated(subset=['train_no','date','station']).sum() == 0

    
'''[ASSERT] weather columns are not null post-join '''
def assert_weather_col(df_merged,df_w):
    df_weather_col = [col for col in df_w.columns if col not in ('date','train_no')]
    assert df_merged[df_weather_col].isna().sum().sum() == 0,(
        "Mismatch \n",
        f"{df_merged[df_weather_col].isna().sum()}"
    )
assert_lossless_join
assert_weather_col(df,df_w)
assert_dates(train,weather)

In [ ]:
''' 
Descriptive — which trains/routes/stations are worst overall
Temporal — day of week, month, season patterns
Weather correlation — rain/fog vs delay magnitude
Cascading analysis — does delay grow along the route?
Combined — a simple regression or feature importance showing which factor matters most
'''

In [34]:
df_ts = pd.read_csv("../data/processed/time_series.csv")
df_w = pd.read_csv("../data/processed/weather.csv")

In [35]:
# df_ts.loc[len(df_ts)] = ['15960','2025-05-03','DBRG','2.0']

In [36]:
df_ts = df_ts.rename(columns={"Train": "train_no", "Date": "date", "Station": "station", "Delay": "delay_minutes"})
df_ts["date"] = pd.to_datetime(df_ts["date"])
df_w["date"] = pd.to_datetime(df_w["date"])

In [37]:
df_ts.head()

,train_no,date,station,delay_minutes
0,15960,2025-05-02,DBRG,1.0
1,15960,2025-05-03,DBRG,0.0
2,15960,2025-05-05,DBRG,1.0
3,15960,2025-05-06,DBRG,0.0
4,15960,2025-05-07,DBRG,1.0


In [38]:
df_ts.shape

(120757, 4)

In [40]:
print(df_ts['date'].min(), df_ts['date'].max())
print(df_w['date'].min(), df_w['date'].max())

2025-05-01 00:00:00 2026-04-30 00:00:00
2025-05-01 00:00:00 2026-04-30 00:00:00


In [41]:
df_ts.nunique()

train_no           9
date             365
station          216
delay_minutes    836
dtype: int64

In [42]:
print(df_w.shape)
print(df_ts.shape)

(3279, 11)
(120757, 4)


In [43]:
df_w.sample(5)

,train_no,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,wind_speed_10m_max,wind_gusts_10m_max,relative_humidity_2m_mean,weather_code
1801,12423,2026-04-09,22.8,17.5,19.6,7.9,7.9,8.9,18.7,87,63
1972,15946,2025-09-30,34.4,24.5,29.3,24.9,24.9,20.3,36.4,79,65
2303,15657,2025-08-28,32.7,25.0,28.7,27.9,27.9,8.6,22.3,86,65
922,15615,2025-11-11,28.0,19.5,23.7,1.0,1.0,9.5,18.4,83,51
748,15615,2025-05-21,29.9,23.1,25.9,7.1,7.1,7.1,24.5,89,61


In [44]:
df_ts.info()

<class 'pandas.DataFrame'>
RangeIndex: 120757 entries, 0 to 120756
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   train_no       120757 non-null  int64         
 1   date           120757 non-null  datetime64[us]
 2   station        120757 non-null  str           
 3   delay_minutes  108412 non-null  float64       
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 3.7 MB


In [45]:
df = df_ts.merge(df_w,on=['date','train_no'],how='inner')
df.head()

,train_no,date,station,delay_minutes,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,wind_speed_10m_max,wind_gusts_10m_max,relative_humidity_2m_mean,weather_code
0,15960,2025-05-02,DBRG,1.0,28.5,20.7,24.7,1.3,1.3,15.9,34.6,76,61
1,15960,2025-05-03,DBRG,0.0,29.4,20.7,24.7,7.7,7.7,6.4,18.7,82,63
2,15960,2025-05-05,DBRG,1.0,30.2,21.2,25.5,12.2,12.2,12.5,23.4,82,63
3,15960,2025-05-06,DBRG,0.0,30.4,21.4,24.7,13.8,13.8,11.5,24.1,85,63
4,15960,2025-05-07,DBRG,1.0,28.4,21.1,23.5,63.2,63.2,12.4,32.4,88,65


In [46]:
df.sample(5)

,train_no,date,station,delay_minutes,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,wind_speed_10m_max,wind_gusts_10m_max,relative_humidity_2m_mean,weather_code
40624,12423,2026-01-11,DMV,38.0,23.4,11.7,16.8,0.0,0.0,6.6,16.2,73,1
21342,15615,2026-03-15,JMK,63.0,23.6,18.2,20.5,13.5,13.5,11.5,30.6,88,63
44693,12423,2026-03-06,KIR,25.0,30.9,17.5,24.2,0.0,0.0,9.1,15.1,66,3
51524,15946,2025-05-21,BARH,65.0,29.4,22.2,25.3,2.4,2.4,11.7,27.0,86,53
81430,12505,2025-08-02,SFG,NaN,30.2,25.6,27.8,29.9,29.9,8.6,22.7,91,63


In [47]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120757 entries, 0 to 120756
Data columns (total 13 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   train_no                   120757 non-null  int64         
 1   date                       120757 non-null  datetime64[us]
 2   station                    120757 non-null  str           
 3   delay_minutes              108412 non-null  float64       
 4   temperature_2m_max         120757 non-null  float64       
 5   temperature_2m_min         120757 non-null  float64       
 6   temperature_2m_mean        120757 non-null  float64       
 7   precipitation_sum          120757 non-null  float64       
 8   rain_sum                   120757 non-null  float64       
 9   wind_speed_10m_max         120757 non-null  float64       
 10  wind_gusts_10m_max         120757 non-null  float64       
 11  relative_humidity_2m_mean  120757 non-null  int64         
 12 

In [48]:
print(f"{df.shape} {df_ts.shape} {df_w.shape}") 

(120757, 13) (120757, 4) (3279, 11)


In [ ]:

''' [ASSERT] a lossless merge of two dataset '''
def assert_lossless_join(df_merged,df_ts):
    #row mismatch
    assert df_merged.shape[0] ==  df_ts.shape[0]
    #duplicate rows
    assert df_merged.duplicated(subset=['train_no','date','station']).sum() == 0

    
'''[ASSERT] weather columns are not null post-join '''
def assert_weather_col(df_merged,df_w):
    df_weather_col = [col for col in df_w.columns if col not in ('date','train_no')]
    assert df_merged[df_weather_col].isna().sum().sum() == 0,(
        "Mismatch \n",
        f"{df_merged[df_weather_col].isna().sum()}"
    )
assert_weather_col(df,df_w)

In [50]:
''' [CAUTION] each row is a station info not a single day

train_no | date       | station
15960    | 2025-04-28 | DBRG
15960    | 2025-04-28 | (another station...)
...
15960    | 2026-04-25 | HWH

each date can have more than 1 station so row number > 365 

'''
df[df['train_no']==15960]

,train_no,date,station,delay_minutes,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,wind_speed_10m_max,wind_gusts_10m_max,relative_humidity_2m_mean,weather_code
0,15960,2025-05-02,DBRG,1.0,28.5,20.7,24.7,1.3,1.3,15.9,34.6,76,61
1,15960,2025-05-03,DBRG,0.0,29.4,20.7,24.7,7.7,7.7,6.4,18.7,82,63
2,15960,2025-05-05,DBRG,1.0,30.2,21.2,25.5,12.2,12.2,12.5,23.4,82,63
3,15960,2025-05-06,DBRG,0.0,30.4,21.4,24.7,13.8,13.8,11.5,24.1,85,63
4,15960,2025-05-07,DBRG,1.0,28.4,21.1,23.5,63.2,63.2,12.4,32.4,88,65
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15855,15960,2026-04-24,HWH,-10.0,32.3,22.5,27.7,5.4,5.4,13.3,24.1,72,63
15856,15960,2026-04-25,HWH,195.0,27.8,20.8,23.7,8.1,8.1,13.2,31.7,86,61
15857,15960,2026-04-27,HWH,-9.0,23.4,20.4,21.8,12.9,12.9,11.4,23.4,93,63
15858,15960,2026-04-28,HWH,74.0,28.7,21.0,24.7,5.4,5.4,13.0,28.1,80,63


In [51]:
df[df['delay_minutes']<0]

,train_no,date,station,delay_minutes,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,wind_speed_10m_max,wind_gusts_10m_max,relative_humidity_2m_mean,weather_code
15601,15960,2025-05-03,HWH,-36.0,29.4,20.7,24.7,7.7,7.7,6.4,18.7,82,63
15604,15960,2025-05-07,HWH,-40.0,28.4,21.1,23.5,63.2,63.2,12.4,32.4,88,65
15605,15960,2025-05-09,HWH,-21.0,31.5,22.8,27.0,2.5,2.5,9.9,22.3,79,61
15606,15960,2025-05-10,HWH,-14.0,31.5,23.6,27.3,5.3,5.3,16.1,30.6,81,63
15608,15960,2025-05-13,HWH,-42.0,28.3,22.0,24.3,38.1,38.1,14.2,30.6,90,65
...,...,...,...,...,...,...,...,...,...,...,...,...,...
120698,15909,2026-03-03,LGH,-10.0,27.2,15.2,21.1,0.0,0.0,7.7,20.5,70,1
120707,15909,2026-03-12,LGH,-15.0,25.4,19.4,21.9,3.3,3.3,11.7,27.7,83,61
120718,15909,2026-03-23,LGH,-10.0,24.0,18.1,20.3,5.6,5.6,8.4,21.6,85,61
120743,15909,2026-04-17,LGH,-12.0,30.7,20.8,25.9,0.0,0.0,7.0,15.8,74,2


In [52]:
#[WARN]
'''0.0 -> no delay
NULL -> missing / not recorded'''
mising_pct = df.delay_minutes.isnull().sum()/df.shape[0] * 100
# print(type(mising_pct))
print(f"{np.round(mising_pct,2)}%")

10.22%


In [53]:
df.shape

(120757, 13)

In [54]:
#How much data do we have 
result = duckdb.sql(
    """select
    train_no,
    count(*) as total_records,
    MIN(date) as earliest_date,
    MAX(date) as latest_date,
    COUNT(distinct date ) as days_covered
    from df
    group by train_no
    order by days_covered
    """  
).to_df()
result

,train_no,total_records,earliest_date,latest_date,days_covered
0,15946,5928,2025-05-04,2026-04-29,104
1,15960,15860,2025-05-02,2026-04-29,260
2,12067,4069,2025-05-01,2026-04-30,313
3,12505,12775,2025-05-01,2026-04-30,365
4,15909,36135,2025-05-01,2026-04-30,365
5,15615,12410,2025-05-01,2026-04-30,365
6,15657,17520,2025-05-01,2026-04-30,365
7,12346,6570,2025-05-01,2026-04-30,365
8,12423,9490,2025-05-01,2026-04-30,365


In [55]:
#check for nulls in delays per train
result = duckdb.sql("""
select
    train_no,
    Count(*) total_rows,
    Count(delay_minutes) as non_null_delays,
    Count(*) - Count(delay_minutes) as missing_count,
    Round(
        (Count(*)-Count(delay_minutes))/Count(*)*100.0
        ,2) as missing_pct
    from df 
    group by train_no,
    order by missing_pct
                    """).to_df()
result

,train_no,total_rows,non_null_delays,missing_count,missing_pct
0,15946,5928,5747,181,3.05
1,15960,15860,14785,1075,6.78
2,15657,17520,16275,1245,7.11
3,12423,9490,8739,751,7.91
4,12346,6570,5958,612,9.32
5,15909,36135,31768,4367,12.09
6,15615,12410,10898,1512,12.18
7,12067,4069,3528,541,13.30
8,12505,12775,10714,2061,16.13


In [56]:
''' 
[WARN] how to better reprenst delay freq if some train have more missing data then other train as 
total missing is 10% individual trains pct differ
(Minimal sol) Create a weighted score
    high delay + high data -> high score
    high delay + low data -> reduced score
'''
#3.most delayed trains ranking 

result = duckdb.sql("""
    select
        train_no,
        Round(AVG(delay_minutes),2) as avg_delay,
        MAX(delay_minutes) as worst_delay,
        Count(Distinct date) as total_days_running,
        -- counting distinct days when delay > 10min
        Count(
            Distinct Case When delay_minutes>10 then date end)as days_delayed,
        ROUND(
        (COUNT(DISTINCT CASE WHEN delay_minutes > 10 THEN date END) * 1.0
        / COUNT(DISTINCT CASE WHEN delay_minutes IS NOT NULL THEN date END))
        *
        (COUNT(DISTINCT CASE WHEN delay_minutes IS NOT NULL THEN date END) * 1.0
        / COUNT(DISTINCT date))
        * 100,
    2) AS adjusted_delay_score_freq
    from df,
    group by train_no 
    order by adjusted_delay_score_freq DESC , avg_delay DESC
""").to_df()
result

,train_no,avg_delay,worst_delay,total_days_running,days_delayed,adjusted_delay_score_freq
0,15657,77.60,744.0,365,365,100.00
1,15946,41.40,703.0,104,104,100.00
2,15960,38.34,963.0,260,260,100.00
3,12423,32.23,696.0,365,365,100.00
4,12346,29.92,503.0,365,362,99.18
5,15615,64.84,2949.0,365,354,96.99
6,15909,76.83,970.0,365,352,96.44
7,12505,39.48,732.0,365,340,93.15
8,12067,9.49,113.0,313,267,85.30


In [57]:
# Monthly delay trend
result = duckdb.sql("""
    select
        date_trunc('month',date) as month,
        Round(AVG(delay_minutes),2) as avg_delay,       
        COUNT(DISTINCT train_no) AS trains_running
    FROM df
    WHERE delay_minutes IS NOT NULL
    GROUP BY month
    ORDER BY month;
""").to_df()
result

,month,avg_delay,trains_running
0,2025-05-01,40.86,9
1,2025-06-01,39.02,9
2,2025-07-01,42.74,9
3,2025-08-01,31.72,9
4,2025-09-01,45.87,9
5,2025-10-01,61.34,9
6,2025-11-01,58.75,9
7,2025-12-01,111.73,9
8,2026-01-01,92.43,9
9,2026-02-01,52.63,9


In [58]:
#delay buckets
#without daily_dely we we counting station on same day mor then once 
result = duckdb.sql("""
                    
WITH daily_delay AS (
    SELECT
        train_no,
        date,
        MAX(delay_minutes) AS max_delay,
        MIN(delay_minutes) AS min_delay
    FROM df
    WHERE delay_minutes IS NOT NULL
    GROUP BY train_no, date
)
SELECT
    train_no,
    COUNT(CASE WHEN min_delay < 0 THEN 1 END) AS early_days,
    COUNT(CASE WHEN max_delay <= 10 THEN 1 END) AS on_time,
    COUNT(CASE WHEN max_delay > 10 AND max_delay <= 30 THEN 1 END) AS minor_delay,
    COUNT(CASE WHEN max_delay > 30 AND max_delay <= 120 THEN 1 END) AS moderate_delay,
    COUNT(CASE WHEN max_delay > 120 THEN 1 END) AS severe_delay

FROM daily_delay
GROUP BY train_no
ORDER BY severe_delay DESC;
 """).to_df()
result

,train_no,early_days,on_time,minor_delay,moderate_delay,severe_delay
0,15909,51,0,0,121,231
1,15657,236,0,8,249,108
2,12423,83,0,56,237,72
3,15615,112,11,4,280,70
4,12505,101,0,1,274,65
5,12346,105,3,142,172,48
6,15960,156,0,2,215,43
7,15946,18,0,0,72,32
8,12067,235,45,203,64,0
